# Prepare fixed 70:10:20 subsets for the DR mini-project

**Run this notebook once before training or evaluation.** It creates one saved manifest for all three official TFDS splits. All four model trainers and the evaluator must consume this same manifest; they must not sample independently.

The training subset keeps at most 2,500 examples per class. Validation and test are sampled proportionally by class from their respective official splits. Their target sizes are calculated from the retained training count to approximate **70% train / 10% validation / 20% test**; integer rounding means the final percentages may differ slightly.

- Original data in `data/btgraham-300/` is never modified.
- Original official split membership is preserved; selected samples never cross split boundaries.
- Images are not decoded during the label/index scan.
- The manifest is saved under `manifests/`, outside the ignored `data/` folder.
- Metrics represent the fixed sampled validation/test subsets, not the complete official validation/test splits.
- Do not regenerate the manifest between model runs unless intentionally starting a new comparison cohort.


In [1]:
from configs import tensorflow_setup as _tensorflow_setup

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

_tensorflow_setup.configure_gpu_memory_growth(tf)

import tensorflow_datasets as tfds

from configs.config import (
    DATA_DIR, EXPECTED_DATASET_NAME, NUM_CLASSES, TRAIN_SPLIT,
    VALIDATION_SPLIT, TEST_SPLIT, CLASS_NAMES,
    UNDERSAMPLE_MAX_PER_CLASS, UNDERSAMPLE_SEED,
    UNDERSAMPLING_MANIFEST_PATH,
)

# Keep this notebook self-contained: these ratios need not be defined in config.py.
TRAIN_RATIO = 70
VALIDATION_RATIO = 10
TEST_RATIO = 20
if min(TRAIN_RATIO, VALIDATION_RATIO, TEST_RATIO) <= 0:
    raise ValueError("All split ratios must be positive integers.")
if TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO != 100:
    raise ValueError("This notebook expects ratios that sum to 100 (currently 70:10:20).")
if NUM_CLASSES != len(CLASS_NAMES):
    raise ValueError(f"NUM_CLASSES={NUM_CLASSES} but {len(CLASS_NAMES)} class names are configured.")
if UNDERSAMPLE_MAX_PER_CLASS < 1:
    raise ValueError("UNDERSAMPLE_MAX_PER_CLASS must be at least 1.")

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("Dataset:", DATA_DIR)
print("Per-class training cap:", UNDERSAMPLE_MAX_PER_CLASS)
print("Seed:", UNDERSAMPLE_SEED)
print("Target ratio:", f"{TRAIN_RATIO}:{VALIDATION_RATIO}:{TEST_RATIO}")
print("Manifest:", UNDERSAMPLING_MANIFEST_PATH)

Python: 3.10.20
TensorFlow: 2.10.1
TFDS: 4.8.3
Dataset: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\btgraham-300
Per-class training cap: 2500
Seed: 42
Target ratio: 70:10:20
Manifest: D:\M L\Diabetic_Retinopathy_Detection_mini_project\manifests\undersampling\train_indices_seed42_cap2500.json


In [2]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Local btgraham-300 dataset not found: {DATA_DIR}")

required_metadata = ("dataset_info.json", "features.json")
missing_metadata = [name for name in required_metadata if not (DATA_DIR / name).is_file()]
if missing_metadata:
    raise FileNotFoundError(f"Missing TFDS metadata files in {DATA_DIR}: {missing_metadata}")

builder = tfds.builder_from_directory(str(DATA_DIR))
if builder.name != EXPECTED_DATASET_NAME:
    raise ValueError(f"Expected {EXPECTED_DATASET_NAME!r}, found {builder.name!r}")
if "label" not in builder.info.features:
    raise ValueError("The local TFDS dataset has no 'label' feature.")

role_to_split = {
    "train": TRAIN_SPLIT,
    "validation": VALIDATION_SPLIT,
    "test": TEST_SPLIT,
}
for split_name in role_to_split.values():
    if split_name not in builder.info.splits:
        raise ValueError(f"Required split {split_name!r} is missing")

official_sizes = {
    role: int(builder.info.splits[split_name].num_examples)
    for role, split_name in role_to_split.items()
}
if any(count <= 0 for count in official_sizes.values()):
    raise ValueError(f"All three source splits must be non-empty: {official_sizes}")
print("Dataset version:", builder.info.version)
print("Official sizes:")
for role, count in official_sizes.items():
    print(f"  {role:10s}: {count:,}")

Dataset version: 3.0.0
Official sizes:
  train     : 35,126
  validation: 10,906
  test      : 42,670


In [3]:
# Read labels without decoding retinal images. TFDS stream order is fixed with shuffle_files=False.
def read_split_labels(split_name):
    dataset = builder.as_dataset(
        split=split_name,
        shuffle_files=False,
        read_config=tfds.ReadConfig(try_autocache=False),
        decoders={"image": tfds.decode.SkipDecoding()},
    )
    labels = []
    for item in dataset:
        label = int(item["label"].numpy())
        if label < 0 or label >= NUM_CLASSES:
            raise ValueError(f"Out-of-range label {label} in split {split_name}")
        labels.append(label)
    if len(labels) != int(builder.info.splits[split_name].num_examples):
        raise RuntimeError(f"Label count mismatch for split {split_name}")
    return np.asarray(labels, dtype=np.int32)

labels_by_role = {
    role: read_split_labels(split_name)
    for role, split_name in role_to_split.items()
}
original_class_counts = {
    role: np.bincount(labels, minlength=NUM_CLASSES).astype(int).tolist()
    for role, labels in labels_by_role.items()
}
print("Original class counts:")
for role in role_to_split:
    print(f"  {role:10s}: {original_class_counts[role]}")
    if sum(original_class_counts[role]) != official_sizes[role]:
        raise RuntimeError(f"Class counts do not sum to the official {role} split size.")
    missing_classes = [class_id for class_id, count in enumerate(original_class_counts[role]) if count == 0]
    if missing_classes:
        print(f"  Warning: {role} has no examples for class IDs {missing_classes}.")

Original class counts:
  train     : [25810, 2443, 5292, 873, 708]
  validation: [8130, 720, 1579, 237, 240]
  test      : [31403, 3042, 6282, 977, 966]


In [4]:
# Allocate an exact target count proportionally by class using the largest-remainder method.
def proportional_class_allocation(class_counts, target_count):
    counts = np.asarray(class_counts, dtype=np.int64)
    if target_count > int(counts.sum()):
        raise ValueError("Target count is larger than the source split")
    if target_count < 0:
        raise ValueError("Target count cannot be negative")
    if counts.ndim != 1 or len(counts) != NUM_CLASSES:
        raise ValueError(f"Expected {NUM_CLASSES} class counts, got shape {counts.shape}")
    if np.any(counts < 0):
        raise ValueError("Class counts cannot be negative")
    if int(counts.sum()) == 0:
        if target_count == 0:
            return [0] * NUM_CLASSES
        raise ValueError("Cannot allocate examples from an empty source split")
    quotas = target_count * counts / counts.sum()
    allocated = np.floor(quotas).astype(np.int64)
    remaining = int(target_count - allocated.sum())
    # Deterministic tie-break: smaller class id first.
    order = sorted(range(len(counts)), key=lambda i: (-(quotas[i] - allocated[i]), i))
    for class_id in order:
        if remaining == 0:
            break
        if allocated[class_id] < counts[class_id]:
            allocated[class_id] += 1
            remaining -= 1
    if remaining != 0 or int(allocated.sum()) != target_count:
        raise RuntimeError("Could not allocate the target split count exactly")
    return allocated.astype(int).tolist()

# Training: independently cap each class at 2,500 examples.
rng_train = np.random.RandomState(int(UNDERSAMPLE_SEED))
train_indices_by_class = []
for class_id in range(NUM_CLASSES):
    indices = np.flatnonzero(labels_by_role["train"] == class_id)
    keep = min(len(indices), int(UNDERSAMPLE_MAX_PER_CLASS))
    train_indices_by_class.append(rng_train.choice(indices, size=keep, replace=False).astype(int).tolist())
train_indices = sorted(i for group in train_indices_by_class for i in group)
train_class_counts = [len(group) for group in train_indices_by_class]
train_count = len(train_indices)

# Validation/test counts are derived from the retained train count to approximate 70:10:20.
if train_count <= 0:
    raise RuntimeError("The selected training subset is empty.")
validation_target = int(round(train_count * VALIDATION_RATIO / TRAIN_RATIO))
test_target = int(round(train_count * TEST_RATIO / TRAIN_RATIO))
if validation_target < 1 or test_target < 1:
    raise ValueError(
        f"Ratio-derived targets must be non-zero; got validation={validation_target}, test={test_target}."
    )
if validation_target > official_sizes["validation"] or test_target > official_sizes["test"]:
    raise ValueError(
        "The configured ratios request more validation/test examples than available in the official splits."
    )

# Retain original class proportions within validation and test while hitting the exact target totals.
def sample_stratified_indices(role, target_count, seed_offset):
    labels = labels_by_role[role]
    source_counts = np.bincount(labels, minlength=NUM_CLASSES).astype(int).tolist()
    allocation = proportional_class_allocation(source_counts, target_count)
    rng = np.random.RandomState(int(UNDERSAMPLE_SEED) + seed_offset)
    chosen_by_class = []
    for class_id, keep in enumerate(allocation):
        indices = np.flatnonzero(labels == class_id)
        if keep > len(indices):
            raise RuntimeError(f"Cannot retain {keep} class-{class_id} examples from {role}")
        chosen = rng.choice(indices, size=keep, replace=False) if keep else np.asarray([], dtype=int)
        chosen_by_class.append(chosen.astype(int).tolist())
    return sorted(i for group in chosen_by_class for i in group), allocation

validation_indices, validation_class_counts = sample_stratified_indices("validation", validation_target, 1)
test_indices, test_class_counts = sample_stratified_indices("test", test_target, 2)

selected = {
    "train": (train_indices, train_class_counts),
    "validation": (validation_indices, validation_class_counts),
    "test": (test_indices, test_class_counts),
}
for role, (indices, counts) in selected.items():
    if len(indices) != len(set(indices)):
        raise RuntimeError(f"Duplicate indices in {role}")
    if indices != sorted(indices):
        raise RuntimeError(f"Indices must be sorted for {role}")
    if any(index < 0 or index >= official_sizes[role] for index in indices):
        raise RuntimeError(f"An index falls outside the official {role} split")
    if len(indices) != sum(counts):
        raise RuntimeError(f"Class counts do not sum to selected examples for {role}")
    observed_counts = np.bincount(labels_by_role[role][indices], minlength=NUM_CLASSES).astype(int).tolist()
    if observed_counts != [int(x) for x in counts]:
        raise RuntimeError(f"Selected class counts do not match labels for {role}: {observed_counts} != {counts}")

In [5]:
# Save all three splits in a single manifest. Each split has its own index checksum;
# the complete manifest also has a checksum recorded in every model's metadata.
def indices_sha256(indices):
    raw = json.dumps([int(x) for x in indices], separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

splits_manifest = {}
for role, split_name in role_to_split.items():
    indices, selected_counts = selected[role]
    splits_manifest[role] = {
        "split_name": split_name,
        "official_count": int(official_sizes[role]),
        "original_class_counts": original_class_counts[role],
        "retained_count": len(indices),
        "retained_class_counts": [int(x) for x in selected_counts],
        "indices_sha256": indices_sha256(indices),
        "indices": [int(x) for x in indices],
    }

manifest = {
    "manifest_version": 2,
    "dataset_name": builder.name,
    "dataset_version": str(builder.info.version),
    "source_dataset_format": "btgraham-300",
    "sampling_method": {
        "train": "random_without_replacement_with_max_per_class",
        "validation": "stratified_random_without_replacement_to_70_10_20_ratio",
        "test": "stratified_random_without_replacement_to_70_10_20_ratio",
    },
    "sampling_seed": int(UNDERSAMPLE_SEED),
    "max_per_class": int(UNDERSAMPLE_MAX_PER_CLASS),
    "split_ratios": {"train": int(TRAIN_RATIO), "validation": int(VALIDATION_RATIO), "test": int(TEST_RATIO)},
    "splits": splits_manifest,
}
unsigned = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode("utf-8")
manifest["split_manifest_sha256"] = hashlib.sha256(unsigned).hexdigest()

# Verify the same checksum calculation used by scripts/data_loader.py.
unsigned_check = dict(manifest)
unsigned_check.pop("split_manifest_sha256", None)
check_bytes = json.dumps(unsigned_check, sort_keys=True, separators=(",", ":")).encode("utf-8")
if hashlib.sha256(check_bytes).hexdigest() != manifest["split_manifest_sha256"]:
    raise RuntimeError("Internal manifest checksum verification failed.")

UNDERSAMPLING_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
with UNDERSAMPLING_MANIFEST_PATH.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)
    file.write("\n")

print("Manifest saved:", UNDERSAMPLING_MANIFEST_PATH)
print("Selected split sizes and ratios:")
total_selected = sum(len(value[0]) for value in selected.values())
for role, (indices, counts) in selected.items():
    pct = 100.0 * len(indices) / total_selected if total_selected else 0.0
    print(f"  {role:10s}: {len(indices):,} ({pct:5.2f}%), classes={counts}")
print("Requested ratio:", f"{TRAIN_RATIO}:{VALIDATION_RATIO}:{TEST_RATIO}")
print("Actual ratio (counts):", f"{len(train_indices)}:{len(validation_indices)}:{len(test_indices)}")
print("Total selected:", f"{total_selected:,}")
print("Whole-manifest SHA-256:", manifest["split_manifest_sha256"])

Manifest saved: D:\M L\Diabetic_Retinopathy_Detection_mini_project\manifests\undersampling\train_indices_seed42_cap2500.json
Selected split sizes and ratios:
  train     : 9,024 (70.00%), classes=[2500, 2443, 2500, 873, 708]
  validation: 1,289 (10.00%), classes=[961, 85, 187, 28, 28]
  test      : 2,578 (20.00%), classes=[1897, 184, 380, 59, 58]
Requested ratio: 70:10:20
Actual ratio (counts): 9024:1289:2578
Total selected: 12,891
Whole-manifest SHA-256: d8a125deedf71c91bc844c3cbe64e0d2a4ee144dcf604093e899a56401d7d471


In [6]:
# Read back and independently validate the saved artifact before model training.
with UNDERSAMPLING_MANIFEST_PATH.open("r", encoding="utf-8") as file:
    saved = json.load(file)

if saved.get("manifest_version") != 2:
    raise RuntimeError(f"Unsupported manifest version: {saved.get('manifest_version')}")
if saved.get("split_manifest_sha256") != manifest.get("split_manifest_sha256"):
    raise RuntimeError("Saved manifest checksum differs from the in-memory manifest.")
unsigned_saved = dict(saved)
recorded_checksum = unsigned_saved.pop("split_manifest_sha256", None)
saved_bytes = json.dumps(unsigned_saved, sort_keys=True, separators=(",", ":")).encode("utf-8")
if hashlib.sha256(saved_bytes).hexdigest() != recorded_checksum:
    raise RuntimeError("Saved manifest checksum verification failed.")
if saved.get("split_ratios") != {"train": TRAIN_RATIO, "validation": VALIDATION_RATIO, "test": TEST_RATIO}:
    raise RuntimeError("Saved split ratios do not match this notebook.")

for role in role_to_split:
    item = saved["splits"][role]
    indices = item["indices"]
    if len(indices) != item["retained_count"]:
        raise RuntimeError(f"Retained count mismatch for {role}.")
    if indices != sorted(set(indices)):
        raise RuntimeError(f"Indices for {role} are not sorted and unique.")
    if any(index < 0 or index >= official_sizes[role] for index in indices):
        raise RuntimeError(f"Index outside official {role} split.")
    if sum(item["retained_class_counts"]) != item["retained_count"]:
        raise RuntimeError(f"Class-count total mismatch for {role}.")
    if item["retained_count"] != len(selected[role][0]):
        raise RuntimeError(f"Saved selected count differs from generated count for {role}.")
    if item["indices_sha256"] != indices_sha256(indices):
        raise RuntimeError(f"Index checksum failed for {role}.")

print("Manifest verification: PASSED")
print("Training examples:", saved["splits"]["train"]["retained_count"])
print("Validation examples:", saved["splits"]["validation"]["retained_count"])
print("Test examples:", saved["splits"]["test"]["retained_count"])
print("Run training.ipynb only after this notebook completes successfully.")


Manifest verification: PASSED
Training examples: 9024
Validation examples: 1289
Test examples: 2578
Run training.ipynb only after this notebook completes successfully.


## Why this improves the deadline situation

- The training subset is capped at 2,500 examples per class (the expected total with the current dataset is 9,024).
- Validation and test are reduced to targets derived from the retained training count and 70:10:20 ratios (expected about 1,289 and 2,578).
- The loader should filter by manifest index before decoding image bytes, then decode only selected examples.
- The same manifest checksum must appear in all undersampled model runs. Do not rerun this notebook between models unless you intentionally want a new comparison cohort.

The exact counts are printed by the notebook and can differ if the local dataset metadata differs from the expected counts. Metrics are on fixed sampled subsets, not the full official validation/test splits; mention this limitation in the report.
